# Test 4 — Temperature and Repeatability

## 1. Purpose

The goal of this test is to understand how the **temperature parameter** affects the behaviour of a language model during text generation.

Temperature controls the **randomness of token selection** during generation. Lower values produce more deterministic outputs, while higher values allow the model to explore more varied responses.

This test investigates how temperature affects:

- response **consistency**
- response **variation**
- likelihood of **hallucination**
- usefulness for **research or explanatory tasks**

The same prompt was executed **multiple times at different temperatures** in order to observe how stable the responses were.

The prompt used was:

> *A researcher wants to use a large language model to help analyse scientific papers. What are three useful ways the model could help, and what are three risks the researcher should be careful about?*

This prompt was chosen because:

- it is a **clear explanatory task**
- it allows repeated runs to be compared easily
- it is relevant to research-support use cases
- it makes it easier to notice drift, repetition, or hallucination as temperature changes

## 2. Setup

This test was executed on the **Bede HPC system** using the `ghtest` GPU partition.

The job ran on node:

`gh008.bede.dur.ac.uk`

with the GPU:

`NVIDIA GH200 144G HBM3e`

as reported in the Slurm job log.

### Fixed parameters

The following parameters were kept constant across all runs:

| Parameter | Value |
|---|---|
| Model | `facebook/opt-125m` |
| GPU count | `1` |
| Tensor parallel size | `1` |
| Max tokens | `300` |
| Prompt | Same for all runs |

### Temperature values tested

Four temperatures were tested, with **three runs per temperature**:

| Temperature | Runs |
|---|---:|
| 0.0 | 3 |
| 0.3 | 3 |
| 0.7 | 3 |
| 1.0 | 3 |

This setup allows observation of **repeatability vs randomness** at each temperature level.

### Important note on interpretation

In this run, the model used was **very small** (`facebook/opt-125m`), and the outputs show that model quality is a major confounding factor. That means this notebook is not only showing the effect of temperature, but also revealing the limitations of using a very small model for research-support tasks.

In [ ]:
##3.Code and Commands

## 3.Code and Commands

In [ ]:
nano 04_temperature_test.sbatch

In [ ]:
#!/bin/bash
#SBATCH --job-name=vllm-test4
#SBATCH --partition=ghtest
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --gres=gpu:1
#SBATCH --time=00:15:00
#SBATCH -A bddur53
#SBATCH --output=vllm_test4_%j.out
#SBATCH --error=vllm_test4_%j.err

set -euo pipefail

echo "===== JOB START ====="
echo "Host: $(hostname)"
echo "Date: $(date)"
echo "Using GPU:"
nvidia-smi

# -----------------------------
# User-editable section
# -----------------------------
PROJECT="bddur53"
USER_NAME="nin6"

BASE="/nobackup/projects/${PROJECT}/${USER_NAME}/modules/vllm26"
CONTAINER_DIR="/nobackup/projects/${PROJECT}/${USER_NAME}/containers"

IMAGE="${CONTAINER_DIR}/vllm-ag2-26.01.1.sif"

HOME_DIR="${BASE}/home"
WORK_DIR="${BASE}/workspace"
CACHE_DIR="${WORK_DIR}/.cache"
PY_FILE="${WORK_DIR}/test4_temperature.py"

MODEL="facebook/opt-125m"
PROMPT="A researcher wants to use a large language model to help analyse scientific papers. What are three useful ways the model could help, and what are three risks the researcher should be careful about?"
MAXTOKENS="300"

# Test 4 values
TEMPERATURE_LIST=(0.0 0.3 0.7 1.0)

# Number of repeated runs per temperature
REPEATS=3
# -----------------------------

if [[ ! -f "$IMAGE" ]]; then
    echo "ERROR: container image $IMAGE not found"
    exit 1
fi

mkdir -p "$HOME_DIR" "$WORK_DIR" "$CACHE_DIR"
chmod -R a+rwx "$BASE"

echo "===== WRITING PYTHON FILE ====="

cat > "$PY_FILE" <<'PYEOF'
from vllm import LLM, SamplingParams
import argparse
import time

parser = argparse.ArgumentParser()
parser.add_argument("--model", required=True)
parser.add_argument("--prompt", required=True)
parser.add_argument("--maxtokens", type=int, required=True)
parser.add_argument("--temperature", type=float, required=True)
parser.add_argument("--repeat", type=int, required=True)
args = parser.parse_args()

print("===== TEST CONFIG =====")
print(f"Model: {args.model}")
print(f"Temperature: {args.temperature}")
print(f"Max tokens: {args.maxtokens}")
print(f"Repeat index: {args.repeat}")
print(f"Prompt: {args.prompt}")
print("=======================")

start_load = time.time()
llm = LLM(model=args.model)
end_load = time.time()

sampling_params = SamplingParams(
    temperature=args.temperature,
    max_tokens=args.maxtokens
)

start_gen = time.time()
outputs = llm.generate([args.prompt], sampling_params)
end_gen = time.time()

print(f"\nModel load time: {end_load - start_load:.2f} seconds")
print(f"Generation time: {end_gen - start_gen:.2f} seconds")

for out in outputs:
    print("\n===== RESPONSE =====\n")
    print(out.outputs[0].text)
    print("\n===== RESPONSE END =====\n")
PYEOF

if [[ ! -s "$PY_FILE" ]]; then
    echo "ERROR: Python file was not created correctly: $PY_FILE"
    exit 1
fi

echo "Python file created:"
ls -l "$PY_FILE"
echo "----- preview -----"
head -n 20 "$PY_FILE"
echo "-------------------"

echo "===== RUNNING TEST 4: TEMPERATURE AND REPEATABILITY ====="
echo "Model: $MODEL"
echo "Prompt: $PROMPT"
echo "Max tokens: $MAXTOKENS"
echo "Temperature values: ${TEMPERATURE_LIST[*]}"
echo "Repeats per temperature: $REPEATS"

for TEMPERATURE in "${TEMPERATURE_LIST[@]}"; do
    for REPEAT in $(seq 1 "$REPEATS"); do
        echo
        echo "===================================================="
        echo "RUNNING TEST WITH temperature=$TEMPERATURE repeat=$REPEAT"
        echo "===================================================="

        apptainer exec --nv \
          --bind "$WORK_DIR:/workspace" \
          --home "$HOME_DIR:/users/$USER_NAME" \
          --env HF_HOME=/workspace/.cache/huggingface \
          --env XDG_CACHE_HOME=/workspace/.cache \
          --env FLASHINFER_WORKSPACE_DIR=/users/$USER_NAME/.cache/flashinfer \
          --env TORCHINDUCTOR_CACHE_DIR=/workspace/.cache/torchinductor \
          --env TRITON_CACHE_DIR=/workspace/.cache/triton \
          --env VLLM_DISABLE_COMPILE=1 \
          "$IMAGE" \
          python /workspace/test4_temperature.py \
            --model "$MODEL" \
            --prompt "$PROMPT" \
            --maxtokens "$MAXTOKENS" \
            --temperature "$TEMPERATURE" \
            --repeat "$REPEAT"
    done
done

echo
echo "===== JOB END ====="
date

In [ ]:
sbatch 04_temperature_test.sbatch

In [ ]:
nano vllm_test4_[Test_ID].out

In [ ]:
 ===== JOB START =====
Host: gh008.bede.dur.ac.uk
Date: Sun 15 Mar 16:11:32 GMT 2026
Using GPU:
Sun Mar 15 16:11:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GH200 144G HBM3e        On  |   00000019:01:00.0 Off |                    0 |
| N/A   33C    P0             86W /  700W |       3MiB / 146831MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|  No running processes found                                                             |
+-----------------------------------------------------------------------------------------+
===== WRITING PYTHON FILE =====
Python file created:
-rw-rw-r-- 1 nin6 bddur53 1189 Mar 15 16:11 /nobackup/projects/bddur53/nin6/modules/vllm26/workspace/test4_temperature.py
----- preview -----
from vllm import LLM, SamplingParams
import argparse
import time

parser = argparse.ArgumentParser()
parser.add_argument("--model", required=True)
parser.add_argument("--prompt", required=True)
parser.add_argument("--maxtokens", type=int, required=True)
parser.add_argument("--temperature", type=float, required=True)
parser.add_argument("--repeat", type=int, required=True)
args = parser.parse_args()

print("===== TEST CONFIG =====")
print(f"Model: {args.model}")
print(f"Temperature: {args.temperature}")
print(f"Max tokens: {args.maxtokens}")
print(f"Repeat index: {args.repeat}")
print(f"Prompt: {args.prompt}")
print("=======================")

-------------------
===== RUNNING TEST 4: TEMPERATURE AND REPEATABILITY =====
Model: facebook/opt-125m
Prompt: A researcher wants to use a large language model to help analyse scientific papers. What are three useful ways the model could help, and what are three risks the researcher should be careful about?
Max tokens: 300
Temperature values: 0.0 0.3 0.7 1.0
Repeats per temperature: 3

====================================================
RUNNING TEST WITH temperature=0.0 repeat=1
====================================================
===== TEST CONFIG =====
Model: facebook/opt-125m
Temperature: 0.0
Max tokens: 300
Repeat index: 1
Prompt: A researcher wants to use a large language model to help analyse scientific papers. What are three useful ways the model could help, and what are three risks the researcher should be careful about?
=======================
INFO 03-15 16:11:46 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'facebook/opt-125m'}
INFO 03-15 16:11:47 [model.py:514] Resolved architecture: OPTForCausalLM
INFO 03-15 16:11:47 [model.py:1661] Using max model len 2048
INFO 03-15 16:11:48 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=16384.
(EngineCore_DP0 pid=922712) INFO 03-15 16:11:48 [core.py:93] Initializing a V1 LLM engine (v0.13.0+faa43dbf.nv26.01) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/
opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parall
el_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutp
utsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_confi
g=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_metrics=False, kv_cache_metrics_sample=0.01, cudagraph_metrics=False, enable_layerwise_
nvtx_tracing=False), seed=0, served_model_name=facebook/opt-125m, enable_prefix_caching=True, enable_chunked_prefill=True, pooler_config=None, compilation_config={'level': None, 'mode': <CompilationMode.VLLM_COMPI
LE: 3>, 'debug_dump_path': None, 'cache_dir': '', 'compile_cache_save_format': 'binary', 'backend': 'inductor', 'custom_ops': ['none'], 'splitting_ops': ['vllm::unified_attention', 'vllm::unified_attention_with_ou
tput', 'vllm::unified_mla_attention', 'vllm::unified_mla_attention_with_output', 'vllm::mamba_mixer2', 'vllm::mamba_mixer', 'vllm::short_conv', 'vllm::linear_attention', 'vllm::plamo2_mamba_mixer', 'vllm::gdn_atte
ntion_core', 'vllm::kda_attention', 'vllm::sparse_attn_indexer'], 'compile_mm_encoder': False, 'compile_sizes': [], 'compile_ranges_split_points': [16384], 'inductor_compile_config': {'enable_auto_functionalized_v
2': False, 'combo_kernels': True, 'benchmark_combo_kernel': True}, 'inductor_passes': {}, 'cudagraph_mode': <CUDAGraphMode.FULL_AND_PIECEWISE: (2, 1)>, 'cudagraph_num_of_warmups': 1, 'cudagraph_capture_sizes': [1,
 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 272, 288, 304, 320, 336, 352, 368, 384, 400, 416, 432, 448,
 464, 480, 496, 512], 'cudagraph_copy_inputs': False, 'cudagraph_specialize_lora': True, 'use_inductor_graph_partition': False, 'pass_config': {'fuse_norm_quant': False, 'fuse_act_quant': False, 'fuse_attn_quant':
 False, 'eliminate_noops': True, 'enable_sp': False, 'fuse_gemm_comms': False, 'fuse_allreduce_rms': False}, 'max_cudagraph_capture_size': 512, 'dynamic_shapes_config': {'type': <DynamicShapesType.BACKED: 'backed'
>, 'evaluate_guards': False}, 'local_cache_dir': None}
(EngineCore_DP0 pid=922712) INFO 03-15 16:11:51 [parallel_state.py:1203] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.16.44.23:47233 backend=nccl
(EngineCore_DP0 pid=922712) INFO 03-15 16:11:51 [parallel_state.py:1411] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=922712) INFO 03-15 16:11:52 [gpu_model_runner.py:3562] Starting to load model facebook/opt-125m...
(EngineCore_DP0 pid=922712) INFO 03-15 16:11:53 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
(EngineCore_DP0 pid=922712) INFO 03-15 16:11:55 [default_loader.py:308] Loading weights took 0.87 seconds
(EngineCore_DP0 pid=922712) INFO 03-15 16:11:55 [gpu_model_runner.py:3659] Model loading took 0.2389 GiB memory and 2.633877 seconds
(EngineCore_DP0 pid=922712) INFO 03-15 16:11:55 [decorators.py:432] Directly load AOT compilation from path /workspace/.cache/vllm/torch_aot_compile/d6c80550895aa1d4cac91b4fb9ef55dfdb1e6d77fa19cbd34edfd
5629fb0d0b1/rank_0_0/model
(EngineCore_DP0 pid=922712) INFO 03-15 16:11:58 [backends.py:643] Using cache directory: /workspace/.cache/vllm/torch_compile_cache/c9c272ba2f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=922712) INFO 03-15 16:11:58 [backends.py:703] Dynamo bytecode transform time: 2.56 s
(EngineCore_DP0 pid=922712) INFO 03-15 16:11:59 [backends.py:261] Cache the graph of compile range (1, 16384) for later use
(EngineCore_DP0 pid=922712) INFO 03-15 16:12:00 [backends.py:278] Compiling a graph for compile range (1, 16384) takes 2.68 s
(EngineCore_DP0 pid=922712) INFO 03-15 16:12:00 [monitor.py:34] torch.compile takes 5.23 s in total
(EngineCore_DP0 pid=922712) INFO 03-15 16:12:01 [gpu_worker.py:375] Available KV cache memory: 126.00 GiB
(EngineCore_DP0 pid=922712) INFO 03-15 16:12:02 [kv_cache_utils.py:1291] GPU KV cache size: 3,669,872 tokens
(EngineCore_DP0 pid=922712) INFO 03-15 16:12:02 [kv_cache_utils.py:1296] Maximum concurrency for 2,048 tokens per request: 1791.93x
(EngineCore_DP0 pid=922712) INFO 03-15 16:12:04 [gpu_model_runner.py:4587] Graph capturing finished in 2 secs, took 0.04 GiB
(EngineCore_DP0 pid=922712) INFO 03-15 16:12:04 [core.py:259] init engine (profile, create kv cache, warmup model) took 9.14 seconds
INFO 03-15 16:12:05 [llm.py:360] Supported tasks: ['generate']

Model load time: 18.31 seconds
Generation time: 0.36 seconds

===== RESPONSE =====



The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help

===== RESPONSE END =====

ERROR 03-15 16:12:05 [core_client.py:606] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.

====================================================
RUNNING TEST WITH temperature=0.0 repeat=2
====================================================
===== TEST CONFIG =====
Model: facebook/opt-125m
Temperature: 0.0
Max tokens: 300
Repeat index: 2
Prompt: A researcher wants to use a large language model to help analyse scientific papers. What are three useful ways the model could help, and what are three risks the researcher should be careful about?
=======================
INFO 03-15 16:12:13 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'facebook/opt-125m'}
INFO 03-15 16:12:14 [model.py:514] Resolved architecture: OPTForCausalLM
INFO 03-15 16:12:14 [model.py:1661] Using max model len 2048
INFO 03-15 16:12:14 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=16384.
(EngineCore_DP0 pid=923091) INFO 03-15 16:12:15 [core.py:93] Initializing a V1 LLM engine (v0.13.0+faa43dbf.nv26.01) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/
opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parall
el_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutp
utsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_confi
g=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_metrics=False, kv_cache_metrics_sample=0.01, cudagraph_metrics=False, enable_layerwise_
nvtx_tracing=False), seed=0, served_model_name=facebook/opt-125m, enable_prefix_caching=True, enable_chunked_prefill=True, pooler_config=None, compilation_config={'level': None, 'mode': <CompilationMode.VLLM_COMPI
LE: 3>, 'debug_dump_path': None, 'cache_dir': '', 'compile_cache_save_format': 'binary', 'backend': 'inductor', 'custom_ops': ['none'], 'splitting_ops': ['vllm::unified_attention', 'vllm::unified_attention_with_ou
tput', 'vllm::unified_mla_attention', 'vllm::unified_mla_attention_with_output', 'vllm::mamba_mixer2', 'vllm::mamba_mixer', 'vllm::short_conv', 'vllm::linear_attention', 'vllm::plamo2_mamba_mixer', 'vllm::gdn_atte
ntion_core', 'vllm::kda_attention', 'vllm::sparse_attn_indexer'], 'compile_mm_encoder': False, 'compile_sizes': [], 'compile_ranges_split_points': [16384], 'inductor_compile_config': {'enable_auto_functionalized_v
2': False, 'combo_kernels': True, 'benchmark_combo_kernel': True}, 'inductor_passes': {}, 'cudagraph_mode': <CUDAGraphMode.FULL_AND_PIECEWISE: (2, 1)>, 'cudagraph_num_of_warmups': 1, 'cudagraph_capture_sizes': [1,
 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 272, 288, 304, 320, 336, 352, 368, 384, 400, 416, 432, 448,
 464, 480, 496, 512], 'cudagraph_copy_inputs': False, 'cudagraph_specialize_lora': True, 'use_inductor_graph_partition': False, 'pass_config': {'fuse_norm_quant': False, 'fuse_act_quant': False, 'fuse_attn_quant':
 False, 'eliminate_noops': True, 'enable_sp': False, 'fuse_gemm_comms': False, 'fuse_allreduce_rms': False}, 'max_cudagraph_capture_size': 512, 'dynamic_shapes_config': {'type': <DynamicShapesType.BACKED: 'backed'
>, 'evaluate_guards': False}, 'local_cache_dir': None}
(EngineCore_DP0 pid=923091) INFO 03-15 16:12:17 [parallel_state.py:1203] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.16.44.23:55925 backend=nccl
(EngineCore_DP0 pid=923091) INFO 03-15 16:12:17 [parallel_state.py:1411] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=923091) INFO 03-15 16:12:17 [gpu_model_runner.py:3562] Starting to load model facebook/opt-125m...
(EngineCore_DP0 pid=923091) INFO 03-15 16:12:18 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
(EngineCore_DP0 pid=923091) INFO 03-15 16:12:19 [default_loader.py:308] Loading weights took 0.07 seconds
(EngineCore_DP0 pid=923091) INFO 03-15 16:12:19 [gpu_model_runner.py:3659] Model loading took 0.2389 GiB memory and 1.180511 seconds
(EngineCore_DP0 pid=923091) INFO 03-15 16:12:19 [decorators.py:432] Directly load AOT compilation from path /workspace/.cache/vllm/torch_aot_compile/d6c80550895aa1d4cac91b4fb9ef55dfdb1e6d77fa19cbd34edfd
5629fb0d0b1/rank_0_0/model
(EngineCore_DP0 pid=923091) INFO 03-15 16:12:22 [backends.py:643] Using cache directory: /workspace/.cache/vllm/torch_compile_cache/c9c272ba2f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=923091) INFO 03-15 16:12:22 [backends.py:703] Dynamo bytecode transform time: 2.44 s
(EngineCore_DP0 pid=923091) INFO 03-15 16:12:22 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 0.482 s
(EngineCore_DP0 pid=923091) INFO 03-15 16:12:22 [monitor.py:34] torch.compile takes 2.92 s in total
(EngineCore_DP0 pid=923091) INFO 03-15 16:12:23 [gpu_worker.py:375] Available KV cache memory: 125.99 GiB
(EngineCore_DP0 pid=923091) INFO 03-15 16:12:23 [kv_cache_utils.py:1291] GPU KV cache size: 3,669,856 tokens
(EngineCore_DP0 pid=923091) INFO 03-15 16:12:23 [kv_cache_utils.py:1296] Maximum concurrency for 2,048 tokens per request: 1791.92x
(EngineCore_DP0 pid=923091) INFO 03-15 16:12:25 [gpu_model_runner.py:4587] Graph capturing finished in 2 secs, took 0.03 GiB
(EngineCore_DP0 pid=923091) INFO 03-15 16:12:25 [core.py:259] init engine (profile, create kv cache, warmup model) took 6.03 seconds
INFO 03-15 16:12:26 [llm.py:360] Supported tasks: ['generate']

Model load time: 12.18 seconds
Generation time: 0.35 seconds

===== RESPONSE =====



The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help

===== RESPONSE END =====

ERROR 03-15 16:12:26 [core_client.py:606] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.

====================================================
RUNNING TEST WITH temperature=0.0 repeat=3
====================================================
===== TEST CONFIG =====
Model: facebook/opt-125m
Temperature: 0.0
Max tokens: 300
Repeat index: 3
Prompt: A researcher wants to use a large language model to help analyse scientific papers. What are three useful ways the model could help, and what are three risks the researcher should be careful about?
=======================
INFO 03-15 16:12:33 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'facebook/opt-125m'}
INFO 03-15 16:12:34 [model.py:514] Resolved architecture: OPTForCausalLM
INFO 03-15 16:12:34 [model.py:1661] Using max model len 2048
INFO 03-15 16:12:34 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=16384.
(EngineCore_DP0 pid=923463) INFO 03-15 16:12:35 [core.py:93] Initializing a V1 LLM engine (v0.13.0+faa43dbf.nv26.01) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/
opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parall
el_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutp
utsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_confi
g=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_metrics=False, kv_cache_metrics_sample=0.01, cudagraph_metrics=False, enable_layerwise_
nvtx_tracing=False), seed=0, served_model_name=facebook/opt-125m, enable_prefix_caching=True, enable_chunked_prefill=True, pooler_config=None, compilation_config={'level': None, 'mode': <CompilationMode.VLLM_COMPI
LE: 3>, 'debug_dump_path': None, 'cache_dir': '', 'compile_cache_save_format': 'binary', 'backend': 'inductor', 'custom_ops': ['none'], 'splitting_ops': ['vllm::unified_attention', 'vllm::unified_attention_with_ou
tput', 'vllm::unified_mla_attention', 'vllm::unified_mla_attention_with_output', 'vllm::mamba_mixer2', 'vllm::mamba_mixer', 'vllm::short_conv', 'vllm::linear_attention', 'vllm::plamo2_mamba_mixer', 'vllm::gdn_atte
ntion_core', 'vllm::kda_attention', 'vllm::sparse_attn_indexer'], 'compile_mm_encoder': False, 'compile_sizes': [], 'compile_ranges_split_points': [16384], 'inductor_compile_config': {'enable_auto_functionalized_v
2': False, 'combo_kernels': True, 'benchmark_combo_kernel': True}, 'inductor_passes': {}, 'cudagraph_mode': <CUDAGraphMode.FULL_AND_PIECEWISE: (2, 1)>, 'cudagraph_num_of_warmups': 1, 'cudagraph_capture_sizes': [1,
 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 272, 288, 304, 320, 336, 352, 368, 384, 400, 416, 432, 448,
 464, 480, 496, 512], 'cudagraph_copy_inputs': False, 'cudagraph_specialize_lora': True, 'use_inductor_graph_partition': False, 'pass_config': {'fuse_norm_quant': False, 'fuse_act_quant': False, 'fuse_attn_quant':
 False, 'eliminate_noops': True, 'enable_sp': False, 'fuse_gemm_comms': False, 'fuse_allreduce_rms': False}, 'max_cudagraph_capture_size': 512, 'dynamic_shapes_config': {'type': <DynamicShapesType.BACKED: 'backed'
>, 'evaluate_guards': False}, 'local_cache_dir': None}
(EngineCore_DP0 pid=923463) INFO 03-15 16:12:36 [parallel_state.py:1203] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.16.44.23:51323 backend=nccl
(EngineCore_DP0 pid=923463) INFO 03-15 16:12:36 [parallel_state.py:1411] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=923463) INFO 03-15 16:12:37 [gpu_model_runner.py:3562] Starting to load model facebook/opt-125m...
(EngineCore_DP0 pid=923463) INFO 03-15 16:12:38 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
(EngineCore_DP0 pid=923463) INFO 03-15 16:12:38 [default_loader.py:308] Loading weights took 0.07 seconds
(EngineCore_DP0 pid=923463) INFO 03-15 16:12:39 [gpu_model_runner.py:3659] Model loading took 0.2389 GiB memory and 1.163769 seconds
(EngineCore_DP0 pid=923463) INFO 03-15 16:12:39 [decorators.py:432] Directly load AOT compilation from path /workspace/.cache/vllm/torch_aot_compile/d6c80550895aa1d4cac91b4fb9ef55dfdb1e6d77fa19cbd34edfd
5629fb0d0b1/rank_0_0/model
(EngineCore_DP0 pid=923463) INFO 03-15 16:12:41 [backends.py:643] Using cache directory: /workspace/.cache/vllm/torch_compile_cache/c9c272ba2f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=923463) INFO 03-15 16:12:41 [backends.py:703] Dynamo bytecode transform time: 2.44 s
(EngineCore_DP0 pid=923463) INFO 03-15 16:12:42 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 0.501 s
(EngineCore_DP0 pid=923463) INFO 03-15 16:12:42 [monitor.py:34] torch.compile takes 2.94 s in total
(EngineCore_DP0 pid=923463) INFO 03-15 16:12:42 [gpu_worker.py:375] Available KV cache memory: 126.00 GiB
(EngineCore_DP0 pid=923463) INFO 03-15 16:12:42 [kv_cache_utils.py:1291] GPU KV cache size: 3,669,888 tokens
(EngineCore_DP0 pid=923463) INFO 03-15 16:12:42 [kv_cache_utils.py:1296] Maximum concurrency for 2,048 tokens per request: 1791.94x
(EngineCore_DP0 pid=923463) INFO 03-15 16:12:45 [gpu_model_runner.py:4587] Graph capturing finished in 2 secs, took 0.03 GiB
(EngineCore_DP0 pid=923463) INFO 03-15 16:12:45 [core.py:259] init engine (profile, create kv cache, warmup model) took 6.02 seconds
INFO 03-15 16:12:45 [llm.py:360] Supported tasks: ['generate']

Model load time: 12.14 seconds
Generation time: 0.34 seconds

===== RESPONSE =====



The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help analyse scientific papers.

The researchers have been working on a new language model that could help

===== RESPONSE END =====

ERROR 03-15 16:12:46 [core_client.py:606] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.

====================================================
RUNNING TEST WITH temperature=0.3 repeat=1
====================================================
===== TEST CONFIG =====
Model: facebook/opt-125m
Temperature: 0.3
Max tokens: 300
Repeat index: 1
Prompt: A researcher wants to use a large language model to help analyse scientific papers. What are three useful ways the model could help, and what are three risks the researcher should be careful about?
=======================
INFO 03-15 16:12:53 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'facebook/opt-125m'}
INFO 03-15 16:12:53 [model.py:514] Resolved architecture: OPTForCausalLM
INFO 03-15 16:12:53 [model.py:1661] Using max model len 2048
INFO 03-15 16:12:54 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=16384.
(EngineCore_DP0 pid=923838) INFO 03-15 16:12:54 [core.py:93] Initializing a V1 LLM engine (v0.13.0+faa43dbf.nv26.01) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/
opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parall
el_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutp
utsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_confi
g=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_metrics=False, kv_cache_metrics_sample=0.01, cudagraph_metrics=False, enable_layerwise_
nvtx_tracing=False), seed=0, served_model_name=facebook/opt-125m, enable_prefix_caching=True, enable_chunked_prefill=True, pooler_config=None, compilation_config={'level': None, 'mode': <CompilationMode.VLLM_COMPI
LE: 3>, 'debug_dump_path': None, 'cache_dir': '', 'compile_cache_save_format': 'binary', 'backend': 'inductor', 'custom_ops': ['none'], 'splitting_ops': ['vllm::unified_attention', 'vllm::unified_attention_with_ou
tput', 'vllm::unified_mla_attention', 'vllm::unified_mla_attention_with_output', 'vllm::mamba_mixer2', 'vllm::mamba_mixer', 'vllm::short_conv', 'vllm::linear_attention', 'vllm::plamo2_mamba_mixer', 'vllm::gdn_atte
ntion_core', 'vllm::kda_attention', 'vllm::sparse_attn_indexer'], 'compile_mm_encoder': False, 'compile_sizes': [], 'compile_ranges_split_points': [16384], 'inductor_compile_config': {'enable_auto_functionalized_v
2': False, 'combo_kernels': True, 'benchmark_combo_kernel': True}, 'inductor_passes': {}, 'cudagraph_mode': <CUDAGraphMode.FULL_AND_PIECEWISE: (2, 1)>, 'cudagraph_num_of_warmups': 1, 'cudagraph_capture_sizes': [1,
 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 272, 288, 304, 320, 336, 352, 368, 384, 400, 416, 432, 448,
 464, 480, 496, 512], 'cudagraph_copy_inputs': False, 'cudagraph_specialize_lora': True, 'use_inductor_graph_partition': False, 'pass_config': {'fuse_norm_quant': False, 'fuse_act_quant': False, 'fuse_attn_quant':
 False, 'eliminate_noops': True, 'enable_sp': False, 'fuse_gemm_comms': False, 'fuse_allreduce_rms': False}, 'max_cudagraph_capture_size': 512, 'dynamic_shapes_config': {'type': <DynamicShapesType.BACKED: 'backed'
>, 'evaluate_guards': False}, 'local_cache_dir': None}
(EngineCore_DP0 pid=923838) INFO 03-15 16:12:56 [parallel_state.py:1203] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.16.44.23:48415 backend=nccl
(EngineCore_DP0 pid=923838) INFO 03-15 16:12:56 [parallel_state.py:1411] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=923838) INFO 03-15 16:12:57 [gpu_model_runner.py:3562] Starting to load model facebook/opt-125m...
(EngineCore_DP0 pid=923838) INFO 03-15 16:12:57 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
(EngineCore_DP0 pid=923838) INFO 03-15 16:12:58 [default_loader.py:308] Loading weights took 0.07 seconds
(EngineCore_DP0 pid=923838) INFO 03-15 16:12:58 [gpu_model_runner.py:3659] Model loading took 0.2389 GiB memory and 1.239017 seconds
(EngineCore_DP0 pid=923838) INFO 03-15 16:12:59 [decorators.py:432] Directly load AOT compilation from path /workspace/.cache/vllm/torch_aot_compile/d6c80550895aa1d4cac91b4fb9ef55dfdb1e6d77fa19cbd34edfd
5629fb0d0b1/rank_0_0/model
(EngineCore_DP0 pid=923838) INFO 03-15 16:13:01 [backends.py:643] Using cache directory: /workspace/.cache/vllm/torch_compile_cache/c9c272ba2f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=923838) INFO 03-15 16:13:01 [backends.py:703] Dynamo bytecode transform time: 2.49 s
(EngineCore_DP0 pid=923838) INFO 03-15 16:13:01 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 0.403 s
(EngineCore_DP0 pid=923838) INFO 03-15 16:13:01 [monitor.py:34] torch.compile takes 2.90 s in total
(EngineCore_DP0 pid=923838) INFO 03-15 16:13:02 [gpu_worker.py:375] Available KV cache memory: 126.00 GiB
(EngineCore_DP0 pid=923838) INFO 03-15 16:13:02 [kv_cache_utils.py:1291] GPU KV cache size: 3,669,872 tokens
(EngineCore_DP0 pid=923838) INFO 03-15 16:13:02 [kv_cache_utils.py:1296] Maximum concurrency for 2,048 tokens per request: 1791.93x
(EngineCore_DP0 pid=923838) INFO 03-15 16:13:04 [gpu_model_runner.py:4587] Graph capturing finished in 2 secs, took 0.04 GiB
(EngineCore_DP0 pid=923838) INFO 03-15 16:13:04 [core.py:259] init engine (profile, create kv cache, warmup model) took 6.03 seconds
INFO 03-15 16:13:05 [llm.py:360] Supported tasks: ['generate']

Model load time: 12.25 seconds
Generation time: 0.35 seconds

===== RESPONSE =====



The researchers have been working on a new language model for the past few years. It is called the “language model”. It is a model that is based on a mathematical model of the universe. It is based on a mathematic
al model of the universe. It is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the

===== RESPONSE END =====

ERROR 03-15 16:13:05 [core_client.py:606] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.

====================================================
RUNNING TEST WITH temperature=0.3 repeat=2
====================================================
===== TEST CONFIG =====
Model: facebook/opt-125m
Temperature: 0.3
Max tokens: 300
Repeat index: 2
Prompt: A researcher wants to use a large language model to help analyse scientific papers. What are three useful ways the model could help, and what are three risks the researcher should be careful about?
=======================
INFO 03-15 16:13:12 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'facebook/opt-125m'}
INFO 03-15 16:13:13 [model.py:514] Resolved architecture: OPTForCausalLM
INFO 03-15 16:13:13 [model.py:1661] Using max model len 2048
INFO 03-15 16:13:13 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=16384.
(EngineCore_DP0 pid=924210) INFO 03-15 16:13:14 [core.py:93] Initializing a V1 LLM engine (v0.13.0+faa43dbf.nv26.01) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/
opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parall
el_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutp
utsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_confi
g=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_metrics=False, kv_cache_metrics_sample=0.01, cudagraph_metrics=False, enable_layerwise_
nvtx_tracing=False), seed=0, served_model_name=facebook/opt-125m, enable_prefix_caching=True, enable_chunked_prefill=True, pooler_config=None, compilation_config={'level': None, 'mode': <CompilationMode.VLLM_COMPI
LE: 3>, 'debug_dump_path': None, 'cache_dir': '', 'compile_cache_save_format': 'binary', 'backend': 'inductor', 'custom_ops': ['none'], 'splitting_ops': ['vllm::unified_attention', 'vllm::unified_attention_with_ou
tput', 'vllm::unified_mla_attention', 'vllm::unified_mla_attention_with_output', 'vllm::mamba_mixer2', 'vllm::mamba_mixer', 'vllm::short_conv', 'vllm::linear_attention', 'vllm::plamo2_mamba_mixer', 'vllm::gdn_atte
ntion_core', 'vllm::kda_attention', 'vllm::sparse_attn_indexer'], 'compile_mm_encoder': False, 'compile_sizes': [], 'compile_ranges_split_points': [16384], 'inductor_compile_config': {'enable_auto_functionalized_v
2': False, 'combo_kernels': True, 'benchmark_combo_kernel': True}, 'inductor_passes': {}, 'cudagraph_mode': <CUDAGraphMode.FULL_AND_PIECEWISE: (2, 1)>, 'cudagraph_num_of_warmups': 1, 'cudagraph_capture_sizes': [1,
 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 272, 288, 304, 320, 336, 352, 368, 384, 400, 416, 432, 448,
 464, 480, 496, 512], 'cudagraph_copy_inputs': False, 'cudagraph_specialize_lora': True, 'use_inductor_graph_partition': False, 'pass_config': {'fuse_norm_quant': False, 'fuse_act_quant': False, 'fuse_attn_quant':
 False, 'eliminate_noops': True, 'enable_sp': False, 'fuse_gemm_comms': False, 'fuse_allreduce_rms': False}, 'max_cudagraph_capture_size': 512, 'dynamic_shapes_config': {'type': <DynamicShapesType.BACKED: 'backed'
>, 'evaluate_guards': False}, 'local_cache_dir': None}
(EngineCore_DP0 pid=924210) INFO 03-15 16:13:16 [parallel_state.py:1203] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.16.44.23:33133 backend=nccl
(EngineCore_DP0 pid=924210) INFO 03-15 16:13:16 [parallel_state.py:1411] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=924210) INFO 03-15 16:13:16 [gpu_model_runner.py:3562] Starting to load model facebook/opt-125m...
(EngineCore_DP0 pid=924210) INFO 03-15 16:13:17 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
(EngineCore_DP0 pid=924210) INFO 03-15 16:13:18 [default_loader.py:308] Loading weights took 0.07 seconds
(EngineCore_DP0 pid=924210) INFO 03-15 16:13:18 [gpu_model_runner.py:3659] Model loading took 0.2389 GiB memory and 1.182462 seconds
(EngineCore_DP0 pid=924210) INFO 03-15 16:13:18 [decorators.py:432] Directly load AOT compilation from path /workspace/.cache/vllm/torch_aot_compile/d6c80550895aa1d4cac91b4fb9ef55dfdb1e6d77fa19cbd34edfd
5629fb0d0b1/rank_0_0/model
(EngineCore_DP0 pid=924210) INFO 03-15 16:13:21 [backends.py:643] Using cache directory: /workspace/.cache/vllm/torch_compile_cache/c9c272ba2f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=924210) INFO 03-15 16:13:21 [backends.py:703] Dynamo bytecode transform time: 2.48 s
(EngineCore_DP0 pid=924210) INFO 03-15 16:13:21 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 0.452 s
(EngineCore_DP0 pid=924210) INFO 03-15 16:13:21 [monitor.py:34] torch.compile takes 2.94 s in total
(EngineCore_DP0 pid=924210) INFO 03-15 16:13:22 [gpu_worker.py:375] Available KV cache memory: 126.00 GiB
(EngineCore_DP0 pid=924210) INFO 03-15 16:13:22 [kv_cache_utils.py:1291] GPU KV cache size: 3,669,872 tokens
(EngineCore_DP0 pid=924210) INFO 03-15 16:13:22 [kv_cache_utils.py:1296] Maximum concurrency for 2,048 tokens per request: 1791.93x
(EngineCore_DP0 pid=924210) INFO 03-15 16:13:24 [gpu_model_runner.py:4587] Graph capturing finished in 2 secs, took 0.04 GiB
(EngineCore_DP0 pid=924210) INFO 03-15 16:13:24 [core.py:259] init engine (profile, create kv cache, warmup model) took 6.05 seconds
INFO 03-15 16:13:25 [llm.py:360] Supported tasks: ['generate']

Model load time: 12.20 seconds
Generation time: 0.35 seconds

===== RESPONSE =====



The researchers have been working on a new language model for the past few years. It is called the “language model”. It is a model that is based on a mathematical model of the universe. It is based on a mathematic
al model of the universe. It is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the

===== RESPONSE END =====

ERROR 03-15 16:13:25 [core_client.py:606] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.

====================================================
RUNNING TEST WITH temperature=0.3 repeat=3
====================================================
===== TEST CONFIG =====
Model: facebook/opt-125m
Temperature: 0.3
Max tokens: 300
Repeat index: 3
Prompt: A researcher wants to use a large language model to help analyse scientific papers. What are three useful ways the model could help, and what are three risks the researcher should be careful about?
=======================
INFO 03-15 16:13:32 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'facebook/opt-125m'}
INFO 03-15 16:13:33 [model.py:514] Resolved architecture: OPTForCausalLM
INFO 03-15 16:13:33 [model.py:1661] Using max model len 2048
INFO 03-15 16:13:33 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=16384.
(EngineCore_DP0 pid=924584) INFO 03-15 16:13:34 [core.py:93] Initializing a V1 LLM engine (v0.13.0+faa43dbf.nv26.01) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/
opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parall
el_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutp
utsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_confi
g=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_metrics=False, kv_cache_metrics_sample=0.01, cudagraph_metrics=False, enable_layerwise_
nvtx_tracing=False), seed=0, served_model_name=facebook/opt-125m, enable_prefix_caching=True, enable_chunked_prefill=True, pooler_config=None, compilation_config={'level': None, 'mode': <CompilationMode.VLLM_COMPI
LE: 3>, 'debug_dump_path': None, 'cache_dir': '', 'compile_cache_save_format': 'binary', 'backend': 'inductor', 'custom_ops': ['none'], 'splitting_ops': ['vllm::unified_attention', 'vllm::unified_attention_with_ou
tput', 'vllm::unified_mla_attention', 'vllm::unified_mla_attention_with_output', 'vllm::mamba_mixer2', 'vllm::mamba_mixer', 'vllm::short_conv', 'vllm::linear_attention', 'vllm::plamo2_mamba_mixer', 'vllm::gdn_atte
ntion_core', 'vllm::kda_attention', 'vllm::sparse_attn_indexer'], 'compile_mm_encoder': False, 'compile_sizes': [], 'compile_ranges_split_points': [16384], 'inductor_compile_config': {'enable_auto_functionalized_v
2': False, 'combo_kernels': True, 'benchmark_combo_kernel': True}, 'inductor_passes': {}, 'cudagraph_mode': <CUDAGraphMode.FULL_AND_PIECEWISE: (2, 1)>, 'cudagraph_num_of_warmups': 1, 'cudagraph_capture_sizes': [1,
 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 272, 288, 304, 320, 336, 352, 368, 384, 400, 416, 432, 448,
 464, 480, 496, 512], 'cudagraph_copy_inputs': False, 'cudagraph_specialize_lora': True, 'use_inductor_graph_partition': False, 'pass_config': {'fuse_norm_quant': False, 'fuse_act_quant': False, 'fuse_attn_quant':
 False, 'eliminate_noops': True, 'enable_sp': False, 'fuse_gemm_comms': False, 'fuse_allreduce_rms': False}, 'max_cudagraph_capture_size': 512, 'dynamic_shapes_config': {'type': <DynamicShapesType.BACKED: 'backed'
>, 'evaluate_guards': False}, 'local_cache_dir': None}
(EngineCore_DP0 pid=924584) INFO 03-15 16:13:35 [parallel_state.py:1203] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.16.44.23:53077 backend=nccl
(EngineCore_DP0 pid=924584) INFO 03-15 16:13:35 [parallel_state.py:1411] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=924584) INFO 03-15 16:13:36 [gpu_model_runner.py:3562] Starting to load model facebook/opt-125m...
(EngineCore_DP0 pid=924584) INFO 03-15 16:13:37 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
(EngineCore_DP0 pid=924584) INFO 03-15 16:13:37 [default_loader.py:308] Loading weights took 0.07 seconds
(EngineCore_DP0 pid=924584) INFO 03-15 16:13:38 [gpu_model_runner.py:3659] Model loading took 0.2389 GiB memory and 1.165963 seconds
(EngineCore_DP0 pid=924584) INFO 03-15 16:13:38 [decorators.py:432] Directly load AOT compilation from path /workspace/.cache/vllm/torch_aot_compile/d6c80550895aa1d4cac91b4fb9ef55dfdb1e6d77fa19cbd34edfd
5629fb0d0b1/rank_0_0/model
(EngineCore_DP0 pid=924584) INFO 03-15 16:13:40 [backends.py:643] Using cache directory: /workspace/.cache/vllm/torch_compile_cache/c9c272ba2f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=924584) INFO 03-15 16:13:40 [backends.py:703] Dynamo bytecode transform time: 2.45 s
(EngineCore_DP0 pid=924584) INFO 03-15 16:13:41 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 0.421 s
(EngineCore_DP0 pid=924584) INFO 03-15 16:13:41 [monitor.py:34] torch.compile takes 2.87 s in total
(EngineCore_DP0 pid=924584) INFO 03-15 16:13:41 [gpu_worker.py:375] Available KV cache memory: 125.99 GiB
(EngineCore_DP0 pid=924584) INFO 03-15 16:13:41 [kv_cache_utils.py:1291] GPU KV cache size: 3,669,856 tokens
(EngineCore_DP0 pid=924584) INFO 03-15 16:13:41 [kv_cache_utils.py:1296] Maximum concurrency for 2,048 tokens per request: 1791.92x
(EngineCore_DP0 pid=924584) INFO 03-15 16:13:44 [gpu_model_runner.py:4587] Graph capturing finished in 2 secs, took 0.04 GiB
(EngineCore_DP0 pid=924584) INFO 03-15 16:13:44 [core.py:259] init engine (profile, create kv cache, warmup model) took 6.00 seconds
INFO 03-15 16:13:44 [llm.py:360] Supported tasks: ['generate']

Model load time: 12.13 seconds
Generation time: 0.35 seconds

===== RESPONSE =====



The researchers have been working on a new language model for the past few years. It is called the “language model”. It is a model that is based on a mathematical model of the universe. It is based on a mathematic
al model of the universe. It is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the universe. It is based on a mathematical model of the universe.

The model is based on a mathematical model of the

===== RESPONSE END =====

ERROR 03-15 16:13:45 [core_client.py:606] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.

====================================================
RUNNING TEST WITH temperature=0.7 repeat=1
====================================================
===== TEST CONFIG =====
Model: facebook/opt-125m
Temperature: 0.7
Max tokens: 300
Repeat index: 1
Prompt: A researcher wants to use a large language model to help analyse scientific papers. What are three useful ways the model could help, and what are three risks the researcher should be careful about?
=======================
INFO 03-15 16:13:52 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'facebook/opt-125m'}
INFO 03-15 16:13:52 [model.py:514] Resolved architecture: OPTForCausalLM
INFO 03-15 16:13:52 [model.py:1661] Using max model len 2048
INFO 03-15 16:13:53 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=16384.
(EngineCore_DP0 pid=924959) INFO 03-15 16:13:53 [core.py:93] Initializing a V1 LLM engine (v0.13.0+faa43dbf.nv26.01) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/
opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parall
el_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutp
utsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_confi
g=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_metrics=False, kv_cache_metrics_sample=0.01, cudagraph_metrics=False, enable_layerwise_
nvtx_tracing=False), seed=0, served_model_name=facebook/opt-125m, enable_prefix_caching=True, enable_chunked_prefill=True, pooler_config=None, compilation_config={'level': None, 'mode': <CompilationMode.VLLM_COMPI
LE: 3>, 'debug_dump_path': None, 'cache_dir': '', 'compile_cache_save_format': 'binary', 'backend': 'inductor', 'custom_ops': ['none'], 'splitting_ops': ['vllm::unified_attention', 'vllm::unified_attention_with_ou
tput', 'vllm::unified_mla_attention', 'vllm::unified_mla_attention_with_output', 'vllm::mamba_mixer2', 'vllm::mamba_mixer', 'vllm::short_conv', 'vllm::linear_attention', 'vllm::plamo2_mamba_mixer', 'vllm::gdn_atte
ntion_core', 'vllm::kda_attention', 'vllm::sparse_attn_indexer'], 'compile_mm_encoder': False, 'compile_sizes': [], 'compile_ranges_split_points': [16384], 'inductor_compile_config': {'enable_auto_functionalized_v
2': False, 'combo_kernels': True, 'benchmark_combo_kernel': True}, 'inductor_passes': {}, 'cudagraph_mode': <CUDAGraphMode.FULL_AND_PIECEWISE: (2, 1)>, 'cudagraph_num_of_warmups': 1, 'cudagraph_capture_sizes': [1,
 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 272, 288, 304, 320, 336, 352, 368, 384, 400, 416, 432, 448,
 464, 480, 496, 512], 'cudagraph_copy_inputs': False, 'cudagraph_specialize_lora': True, 'use_inductor_graph_partition': False, 'pass_config': {'fuse_norm_quant': False, 'fuse_act_quant': False, 'fuse_attn_quant':
 False, 'eliminate_noops': True, 'enable_sp': False, 'fuse_gemm_comms': False, 'fuse_allreduce_rms': False}, 'max_cudagraph_capture_size': 512, 'dynamic_shapes_config': {'type': <DynamicShapesType.BACKED: 'backed'
>, 'evaluate_guards': False}, 'local_cache_dir': None}
(EngineCore_DP0 pid=924959) INFO 03-15 16:13:55 [parallel_state.py:1203] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.16.44.23:45159 backend=nccl
(EngineCore_DP0 pid=924959) INFO 03-15 16:13:55 [parallel_state.py:1411] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=924959) INFO 03-15 16:13:56 [gpu_model_runner.py:3562] Starting to load model facebook/opt-125m...
(EngineCore_DP0 pid=924959) INFO 03-15 16:13:56 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
(EngineCore_DP0 pid=924959) INFO 03-15 16:13:57 [default_loader.py:308] Loading weights took 0.07 seconds
(EngineCore_DP0 pid=924959) INFO 03-15 16:13:57 [gpu_model_runner.py:3659] Model loading took 0.2389 GiB memory and 1.172448 seconds
(EngineCore_DP0 pid=924959) INFO 03-15 16:13:58 [decorators.py:432] Directly load AOT compilation from path /workspace/.cache/vllm/torch_aot_compile/d6c80550895aa1d4cac91b4fb9ef55dfdb1e6d77fa19cbd34edfd
5629fb0d0b1/rank_0_0/model
(EngineCore_DP0 pid=924959) INFO 03-15 16:14:00 [backends.py:643] Using cache directory: /workspace/.cache/vllm/torch_compile_cache/c9c272ba2f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=924959) INFO 03-15 16:14:00 [backends.py:703] Dynamo bytecode transform time: 2.43 s
(EngineCore_DP0 pid=924959) INFO 03-15 16:14:00 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 0.497 s
(EngineCore_DP0 pid=924959) INFO 03-15 16:14:00 [monitor.py:34] torch.compile takes 2.92 s in total
(EngineCore_DP0 pid=924959) INFO 03-15 16:14:01 [gpu_worker.py:375] Available KV cache memory: 126.00 GiB
(EngineCore_DP0 pid=924959) INFO 03-15 16:14:01 [kv_cache_utils.py:1291] GPU KV cache size: 3,669,872 tokens
(EngineCore_DP0 pid=924959) INFO 03-15 16:14:01 [kv_cache_utils.py:1296] Maximum concurrency for 2,048 tokens per request: 1791.93x
(EngineCore_DP0 pid=924959) INFO 03-15 16:14:03 [gpu_model_runner.py:4587] Graph capturing finished in 2 secs, took 0.03 GiB
(EngineCore_DP0 pid=924959) INFO 03-15 16:14:03 [core.py:259] init engine (profile, create kv cache, warmup model) took 6.00 seconds
INFO 03-15 16:14:04 [llm.py:360] Supported tasks: ['generate']

Model load time: 12.09 seconds
Generation time: 0.34 seconds

===== RESPONSE =====



It seems that the chances of writing a paper with a descriptive language are high. Why? Because the language model will use a computer to analyse and analyse the data. What is that language model? Well, you may be
 thinking, because the model is in computer and not the computer. But the computer is not a computer. What is the computer model? Well, what is the computer model? Well, the computer model is a computer that can b
e used to analyse scientific papers. It is a computer that can analyse the scientific papers. To use it, you have to know the computer model and also understand the statistical models. So, a computer is not a comp
uter. It is a computer that can analyse the scientific papers.

The computer is a computer that can analyse the scientific papers. The computer is not a computer. It is not a computer. It is a computer that can analyse the scientific papers. The computer is not a computer. It 
is a computer that can analyse the scientific papers. The computer is not a computer. It is a computer that can analyse the scientific papers. It is a computer that can analyse the scientific papers.

So, you will get one of these kind of computer models. What is a computer model? Well, it will help you in analysing the scientific papers. It is a computer that can analyse the scientific papers. It is a computer
 that can analyse the scientific papers. It is a computer that can analyse the scientific papers. It

===== RESPONSE END =====

ERROR 03-15 16:14:04 [core_client.py:606] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.

====================================================
RUNNING TEST WITH temperature=0.7 repeat=2
====================================================
===== TEST CONFIG =====
Model: facebook/opt-125m
Temperature: 0.7
Max tokens: 300
Repeat index: 2
Prompt: A researcher wants to use a large language model to help analyse scientific papers. What are three useful ways the model could help, and what are three risks the researcher should be careful about?
=======================
INFO 03-15 16:14:11 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'facebook/opt-125m'}
INFO 03-15 16:14:12 [model.py:514] Resolved architecture: OPTForCausalLM
INFO 03-15 16:14:12 [model.py:1661] Using max model len 2048
INFO 03-15 16:14:12 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=16384.
(EngineCore_DP0 pid=925352) INFO 03-15 16:14:13 [core.py:93] Initializing a V1 LLM engine (v0.13.0+faa43dbf.nv26.01) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/
opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parall
el_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutp
utsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_confi
g=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_metrics=False, kv_cache_metrics_sample=0.01, cudagraph_metrics=False, enable_layerwise_
nvtx_tracing=False), seed=0, served_model_name=facebook/opt-125m, enable_prefix_caching=True, enable_chunked_prefill=True, pooler_config=None, compilation_config={'level': None, 'mode': <CompilationMode.VLLM_COMPI
LE: 3>, 'debug_dump_path': None, 'cache_dir': '', 'compile_cache_save_format': 'binary', 'backend': 'inductor', 'custom_ops': ['none'], 'splitting_ops': ['vllm::unified_attention', 'vllm::unified_attention_with_ou
tput', 'vllm::unified_mla_attention', 'vllm::unified_mla_attention_with_output', 'vllm::mamba_mixer2', 'vllm::mamba_mixer', 'vllm::short_conv', 'vllm::linear_attention', 'vllm::plamo2_mamba_mixer', 'vllm::gdn_atte
ntion_core', 'vllm::kda_attention', 'vllm::sparse_attn_indexer'], 'compile_mm_encoder': False, 'compile_sizes': [], 'compile_ranges_split_points': [16384], 'inductor_compile_config': {'enable_auto_functionalized_v
2': False, 'combo_kernels': True, 'benchmark_combo_kernel': True}, 'inductor_passes': {}, 'cudagraph_mode': <CUDAGraphMode.FULL_AND_PIECEWISE: (2, 1)>, 'cudagraph_num_of_warmups': 1, 'cudagraph_capture_sizes': [1,
 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 272, 288, 304, 320, 336, 352, 368, 384, 400, 416, 432, 448,
 464, 480, 496, 512], 'cudagraph_copy_inputs': False, 'cudagraph_specialize_lora': True, 'use_inductor_graph_partition': False, 'pass_config': {'fuse_norm_quant': False, 'fuse_act_quant': False, 'fuse_attn_quant':
 False, 'eliminate_noops': True, 'enable_sp': False, 'fuse_gemm_comms': False, 'fuse_allreduce_rms': False}, 'max_cudagraph_capture_size': 512, 'dynamic_shapes_config': {'type': <DynamicShapesType.BACKED: 'backed'
>, 'evaluate_guards': False}, 'local_cache_dir': None}
(EngineCore_DP0 pid=925352) INFO 03-15 16:14:14 [parallel_state.py:1203] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.16.44.23:54573 backend=nccl
(EngineCore_DP0 pid=925352) INFO 03-15 16:14:15 [parallel_state.py:1411] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=925352) INFO 03-15 16:14:15 [gpu_model_runner.py:3562] Starting to load model facebook/opt-125m...
(EngineCore_DP0 pid=925352) INFO 03-15 16:14:16 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
(EngineCore_DP0 pid=925352) INFO 03-15 16:14:16 [default_loader.py:308] Loading weights took 0.07 seconds
(EngineCore_DP0 pid=925352) INFO 03-15 16:14:17 [gpu_model_runner.py:3659] Model loading took 0.2389 GiB memory and 1.180507 seconds
(EngineCore_DP0 pid=925352) INFO 03-15 16:14:17 [decorators.py:432] Directly load AOT compilation from path /workspace/.cache/vllm/torch_aot_compile/d6c80550895aa1d4cac91b4fb9ef55dfdb1e6d77fa19cbd34edfd
5629fb0d0b1/rank_0_0/model
(EngineCore_DP0 pid=925352) INFO 03-15 16:14:19 [backends.py:643] Using cache directory: /workspace/.cache/vllm/torch_compile_cache/c9c272ba2f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=925352) INFO 03-15 16:14:19 [backends.py:703] Dynamo bytecode transform time: 2.51 s
(EngineCore_DP0 pid=925352) INFO 03-15 16:14:20 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 0.422 s
(EngineCore_DP0 pid=925352) INFO 03-15 16:14:20 [monitor.py:34] torch.compile takes 2.93 s in total
(EngineCore_DP0 pid=925352) INFO 03-15 16:14:20 [gpu_worker.py:375] Available KV cache memory: 126.00 GiB
(EngineCore_DP0 pid=925352) INFO 03-15 16:14:21 [kv_cache_utils.py:1291] GPU KV cache size: 3,669,888 tokens
(EngineCore_DP0 pid=925352) INFO 03-15 16:14:21 [kv_cache_utils.py:1296] Maximum concurrency for 2,048 tokens per request: 1791.94x
(EngineCore_DP0 pid=925352) INFO 03-15 16:14:23 [gpu_model_runner.py:4587] Graph capturing finished in 2 secs, took 0.04 GiB
(EngineCore_DP0 pid=925352) INFO 03-15 16:14:23 [core.py:259] init engine (profile, create kv cache, warmup model) took 6.13 seconds
INFO 03-15 16:14:23 [llm.py:360] Supported tasks: ['generate']

Model load time: 12.29 seconds
Generation time: 0.35 seconds

===== RESPONSE =====



It seems that the chances of writing a paper with a descriptive language are high. Why? Because the language model will use a computer to analyse and analyse the data. What is that language model? Well, you may be
 thinking, because the model is in computer and not the computer. But the computer is not a computer. What is the computer model? Well, what is the computer model? Well, the computer model is a computer that can b
e used to analyse scientific papers. It is a computer that can analyse the scientific papers. To use it, you have to know the computer model and also understand the statistical models. So, a computer is not a comp
uter. It is a computer that can analyse the scientific papers.

The computer is a computer that can analyse the scientific papers. The computer is not a computer. It is not a computer. It is a computer that can analyse the scientific papers. The computer is not a computer. It 
is a computer that can analyse the scientific papers. The computer is not a computer. It is a computer that can analyse the scientific papers. It is a computer that can analyse the scientific papers.

So, you will get one of these kind of computer models. What is a computer model? Well, it will help you in analysing the scientific papers. It is a computer that can analyse the scientific papers. It is a computer
 that can analyse the scientific papers. It is a computer that can analyse the scientific papers. It

===== RESPONSE END =====

ERROR 03-15 16:14:24 [core_client.py:606] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.

====================================================
RUNNING TEST WITH temperature=0.7 repeat=3
====================================================
===== TEST CONFIG =====
Model: facebook/opt-125m
Temperature: 0.7
Max tokens: 300
Repeat index: 3
Prompt: A researcher wants to use a large language model to help analyse scientific papers. What are three useful ways the model could help, and what are three risks the researcher should be careful about?
=======================
INFO 03-15 16:14:31 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'facebook/opt-125m'}
INFO 03-15 16:14:32 [model.py:514] Resolved architecture: OPTForCausalLM
INFO 03-15 16:14:32 [model.py:1661] Using max model len 2048
INFO 03-15 16:14:32 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=16384.
(EngineCore_DP0 pid=925726) INFO 03-15 16:14:33 [core.py:93] Initializing a V1 LLM engine (v0.13.0+faa43dbf.nv26.01) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/
opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parall
el_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutp
utsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_confi
g=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_metrics=False, kv_cache_metrics_sample=0.01, cudagraph_metrics=False, enable_layerwise_
nvtx_tracing=False), seed=0, served_model_name=facebook/opt-125m, enable_prefix_caching=True, enable_chunked_prefill=True, pooler_config=None, compilation_config={'level': None, 'mode': <CompilationMode.VLLM_COMPI
LE: 3>, 'debug_dump_path': None, 'cache_dir': '', 'compile_cache_save_format': 'binary', 'backend': 'inductor', 'custom_ops': ['none'], 'splitting_ops': ['vllm::unified_attention', 'vllm::unified_attention_with_ou
tput', 'vllm::unified_mla_attention', 'vllm::unified_mla_attention_with_output', 'vllm::mamba_mixer2', 'vllm::mamba_mixer', 'vllm::short_conv', 'vllm::linear_attention', 'vllm::plamo2_mamba_mixer', 'vllm::gdn_atte
ntion_core', 'vllm::kda_attention', 'vllm::sparse_attn_indexer'], 'compile_mm_encoder': False, 'compile_sizes': [], 'compile_ranges_split_points': [16384], 'inductor_compile_config': {'enable_auto_functionalized_v
2': False, 'combo_kernels': True, 'benchmark_combo_kernel': True}, 'inductor_passes': {}, 'cudagraph_mode': <CUDAGraphMode.FULL_AND_PIECEWISE: (2, 1)>, 'cudagraph_num_of_warmups': 1, 'cudagraph_capture_sizes': [1,
 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 272, 288, 304, 320, 336, 352, 368, 384, 400, 416, 432, 448,
 464, 480, 496, 512], 'cudagraph_copy_inputs': False, 'cudagraph_specialize_lora': True, 'use_inductor_graph_partition': False, 'pass_config': {'fuse_norm_quant': False, 'fuse_act_quant': False, 'fuse_attn_quant':
 False, 'eliminate_noops': True, 'enable_sp': False, 'fuse_gemm_comms': False, 'fuse_allreduce_rms': False}, 'max_cudagraph_capture_size': 512, 'dynamic_shapes_config': {'type': <DynamicShapesType.BACKED: 'backed'
>, 'evaluate_guards': False}, 'local_cache_dir': None}
(EngineCore_DP0 pid=925726) INFO 03-15 16:14:34 [parallel_state.py:1203] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.16.44.23:59521 backend=nccl
(EngineCore_DP0 pid=925726) INFO 03-15 16:14:34 [parallel_state.py:1411] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=925726) INFO 03-15 16:14:35 [gpu_model_runner.py:3562] Starting to load model facebook/opt-125m...
(EngineCore_DP0 pid=925726) INFO 03-15 16:14:36 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
(EngineCore_DP0 pid=925726) INFO 03-15 16:14:36 [default_loader.py:308] Loading weights took 0.07 seconds
(EngineCore_DP0 pid=925726) INFO 03-15 16:14:36 [gpu_model_runner.py:3659] Model loading took 0.2389 GiB memory and 1.170735 seconds
(EngineCore_DP0 pid=925726) INFO 03-15 16:14:37 [decorators.py:432] Directly load AOT compilation from path /workspace/.cache/vllm/torch_aot_compile/d6c80550895aa1d4cac91b4fb9ef55dfdb1e6d77fa19cbd34edfd
5629fb0d0b1/rank_0_0/model
(EngineCore_DP0 pid=925726) INFO 03-15 16:14:39 [backends.py:643] Using cache directory: /workspace/.cache/vllm/torch_compile_cache/c9c272ba2f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=925726) INFO 03-15 16:14:39 [backends.py:703] Dynamo bytecode transform time: 2.46 s
(EngineCore_DP0 pid=925726) INFO 03-15 16:14:40 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 0.393 s
(EngineCore_DP0 pid=925726) INFO 03-15 16:14:40 [monitor.py:34] torch.compile takes 2.86 s in total
(EngineCore_DP0 pid=925726) INFO 03-15 16:14:40 [gpu_worker.py:375] Available KV cache memory: 126.00 GiB
(EngineCore_DP0 pid=925726) INFO 03-15 16:14:40 [kv_cache_utils.py:1291] GPU KV cache size: 3,669,872 tokens
(EngineCore_DP0 pid=925726) INFO 03-15 16:14:40 [kv_cache_utils.py:1296] Maximum concurrency for 2,048 tokens per request: 1791.93x
(EngineCore_DP0 pid=925726) INFO 03-15 16:14:42 [gpu_model_runner.py:4587] Graph capturing finished in 2 secs, took 0.04 GiB
(EngineCore_DP0 pid=925726) INFO 03-15 16:14:42 [core.py:259] init engine (profile, create kv cache, warmup model) took 6.02 seconds
INFO 03-15 16:14:43 [llm.py:360] Supported tasks: ['generate']

Model load time: 12.08 seconds
Generation time: 0.35 seconds

===== RESPONSE =====



It seems that the chances of writing a paper with a descriptive language are high. Why? Because the language model will use a computer to analyse and analyse the data. What is that language model? Well, you may be
 thinking, because the model is in computer and not the computer. But the computer is not a computer. What is the computer model? Well, what is the computer model? Well, the computer model is a computer that can b
e used to analyse scientific papers. It is a computer that can analyse the scientific papers. To use it, you have to know the computer model and also understand the statistical models. So, a computer is not a comp
uter. It is a computer that can analyse the scientific papers.

The computer is a computer that can analyse the scientific papers. The computer is not a computer. It is not a computer. It is a computer that can analyse the scientific papers. The computer is not a computer. It 
is a computer that can analyse the scientific papers. The computer is not a computer. It is a computer that can analyse the scientific papers. It is a computer that can analyse the scientific papers.

So, you will get one of these kind of computer models. What is a computer model? Well, it will help you in analysing the scientific papers. It is a computer that can analyse the scientific papers. It is a computer
 that can analyse the scientific papers. It is a computer that can analyse the scientific papers. It

===== RESPONSE END =====

ERROR 03-15 16:14:43 [core_client.py:606] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.

====================================================
RUNNING TEST WITH temperature=1.0 repeat=1
====================================================
===== TEST CONFIG =====
Model: facebook/opt-125m
Temperature: 1.0
Max tokens: 300
Repeat index: 1
Prompt: A researcher wants to use a large language model to help analyse scientific papers. What are three useful ways the model could help, and what are three risks the researcher should be careful about?
=======================
INFO 03-15 16:14:51 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'facebook/opt-125m'}
INFO 03-15 16:14:51 [model.py:514] Resolved architecture: OPTForCausalLM
INFO 03-15 16:14:51 [model.py:1661] Using max model len 2048
INFO 03-15 16:14:52 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=16384.
(EngineCore_DP0 pid=926099) INFO 03-15 16:14:52 [core.py:93] Initializing a V1 LLM engine (v0.13.0+faa43dbf.nv26.01) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/
opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parall
el_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutp
utsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_confi
g=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_metrics=False, kv_cache_metrics_sample=0.01, cudagraph_metrics=False, enable_layerwise_
nvtx_tracing=False), seed=0, served_model_name=facebook/opt-125m, enable_prefix_caching=True, enable_chunked_prefill=True, pooler_config=None, compilation_config={'level': None, 'mode': <CompilationMode.VLLM_COMPI
LE: 3>, 'debug_dump_path': None, 'cache_dir': '', 'compile_cache_save_format': 'binary', 'backend': 'inductor', 'custom_ops': ['none'], 'splitting_ops': ['vllm::unified_attention', 'vllm::unified_attention_with_ou
tput', 'vllm::unified_mla_attention', 'vllm::unified_mla_attention_with_output', 'vllm::mamba_mixer2', 'vllm::mamba_mixer', 'vllm::short_conv', 'vllm::linear_attention', 'vllm::plamo2_mamba_mixer', 'vllm::gdn_atte
ntion_core', 'vllm::kda_attention', 'vllm::sparse_attn_indexer'], 'compile_mm_encoder': False, 'compile_sizes': [], 'compile_ranges_split_points': [16384], 'inductor_compile_config': {'enable_auto_functionalized_v
2': False, 'combo_kernels': True, 'benchmark_combo_kernel': True}, 'inductor_passes': {}, 'cudagraph_mode': <CUDAGraphMode.FULL_AND_PIECEWISE: (2, 1)>, 'cudagraph_num_of_warmups': 1, 'cudagraph_capture_sizes': [1,
 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 272, 288, 304, 320, 336, 352, 368, 384, 400, 416, 432, 448,
 464, 480, 496, 512], 'cudagraph_copy_inputs': False, 'cudagraph_specialize_lora': True, 'use_inductor_graph_partition': False, 'pass_config': {'fuse_norm_quant': False, 'fuse_act_quant': False, 'fuse_attn_quant':
 False, 'eliminate_noops': True, 'enable_sp': False, 'fuse_gemm_comms': False, 'fuse_allreduce_rms': False}, 'max_cudagraph_capture_size': 512, 'dynamic_shapes_config': {'type': <DynamicShapesType.BACKED: 'backed'
>, 'evaluate_guards': False}, 'local_cache_dir': None}
(EngineCore_DP0 pid=926099) INFO 03-15 16:14:54 [parallel_state.py:1203] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.16.44.23:58071 backend=nccl
(EngineCore_DP0 pid=926099) INFO 03-15 16:14:54 [parallel_state.py:1411] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=926099) INFO 03-15 16:14:54 [gpu_model_runner.py:3562] Starting to load model facebook/opt-125m...
(EngineCore_DP0 pid=926099) INFO 03-15 16:14:55 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
(EngineCore_DP0 pid=926099) INFO 03-15 16:14:56 [default_loader.py:308] Loading weights took 0.07 seconds
(EngineCore_DP0 pid=926099) INFO 03-15 16:14:56 [gpu_model_runner.py:3659] Model loading took 0.2389 GiB memory and 1.178801 seconds
(EngineCore_DP0 pid=926099) INFO 03-15 16:14:56 [decorators.py:432] Directly load AOT compilation from path /workspace/.cache/vllm/torch_aot_compile/d6c80550895aa1d4cac91b4fb9ef55dfdb1e6d77fa19cbd34edfd
5629fb0d0b1/rank_0_0/model
(EngineCore_DP0 pid=926099) INFO 03-15 16:14:59 [backends.py:643] Using cache directory: /workspace/.cache/vllm/torch_compile_cache/c9c272ba2f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=926099) INFO 03-15 16:14:59 [backends.py:703] Dynamo bytecode transform time: 2.40 s
(EngineCore_DP0 pid=926099) INFO 03-15 16:14:59 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 0.391 s
(EngineCore_DP0 pid=926099) INFO 03-15 16:14:59 [monitor.py:34] torch.compile takes 2.79 s in total
(EngineCore_DP0 pid=926099) INFO 03-15 16:15:00 [gpu_worker.py:375] Available KV cache memory: 126.00 GiB
(EngineCore_DP0 pid=926099) INFO 03-15 16:15:00 [kv_cache_utils.py:1291] GPU KV cache size: 3,669,872 tokens
(EngineCore_DP0 pid=926099) INFO 03-15 16:15:00 [kv_cache_utils.py:1296] Maximum concurrency for 2,048 tokens per request: 1791.93x
(EngineCore_DP0 pid=926099) INFO 03-15 16:15:02 [gpu_model_runner.py:4587] Graph capturing finished in 2 secs, took 0.03 GiB
(EngineCore_DP0 pid=926099) INFO 03-15 16:15:02 [core.py:259] init engine (profile, create kv cache, warmup model) took 5.89 seconds
INFO 03-15 16:15:03 [llm.py:360] Supported tasks: ['generate']

Model load time: 11.94 seconds
Generation time: 0.36 seconds

===== RESPONSE =====

 Joel Schulze explains.

Fautdig runesommer breven der G24bschesinhaber. Et swimförkeln no en diskussion sähte konferenzt:

Månad inntil tegsning föräldrar engagir. Domstamma folgt ett uppgäng enrNUMblokk för vatten – men vilket tal upp jobundersökt i moments dan:väldligen föresterna. Osäkerheten säger:

I kilometer gött billighert på alla lite, samtalen i nämnd arkeologistiskheten. Kan få betydningar? Give betydningministru. De har 8 år till Västerlo Institutet försökt att han går att egna biltamt:nejjande kara s
erioppalets person.

Och som Mo var från Yugoslavia såsom i Europa är det lite att säga att aldrig syvligt än vi somarant till Utveckling ”vat vilka betsägg är.” Vad som har medisinaare bild var?

Fö

===== RESPONSE END =====

ERROR 03-15 16:15:03 [core_client.py:606] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.

====================================================
RUNNING TEST WITH temperature=1.0 repeat=2
====================================================
===== TEST CONFIG =====
Model: facebook/opt-125m
Temperature: 1.0
Max tokens: 300
Repeat index: 2
Prompt: A researcher wants to use a large language model to help analyse scientific papers. What are three useful ways the model could help, and what are three risks the researcher should be careful about?
=======================
INFO 03-15 16:15:10 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'facebook/opt-125m'}
INFO 03-15 16:15:11 [model.py:514] Resolved architecture: OPTForCausalLM
INFO 03-15 16:15:11 [model.py:1661] Using max model len 2048
INFO 03-15 16:15:11 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=16384.
(EngineCore_DP0 pid=926472) INFO 03-15 16:15:12 [core.py:93] Initializing a V1 LLM engine (v0.13.0+faa43dbf.nv26.01) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/
opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parall
el_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutp
utsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_confi
g=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_metrics=False, kv_cache_metrics_sample=0.01, cudagraph_metrics=False, enable_layerwise_
nvtx_tracing=False), seed=0, served_model_name=facebook/opt-125m, enable_prefix_caching=True, enable_chunked_prefill=True, pooler_config=None, compilation_config={'level': None, 'mode': <CompilationMode.VLLM_COMPI
LE: 3>, 'debug_dump_path': None, 'cache_dir': '', 'compile_cache_save_format': 'binary', 'backend': 'inductor', 'custom_ops': ['none'], 'splitting_ops': ['vllm::unified_attention', 'vllm::unified_attention_with_ou
tput', 'vllm::unified_mla_attention', 'vllm::unified_mla_attention_with_output', 'vllm::mamba_mixer2', 'vllm::mamba_mixer', 'vllm::short_conv', 'vllm::linear_attention', 'vllm::plamo2_mamba_mixer', 'vllm::gdn_atte
ntion_core', 'vllm::kda_attention', 'vllm::sparse_attn_indexer'], 'compile_mm_encoder': False, 'compile_sizes': [], 'compile_ranges_split_points': [16384], 'inductor_compile_config': {'enable_auto_functionalized_v
2': False, 'combo_kernels': True, 'benchmark_combo_kernel': True}, 'inductor_passes': {}, 'cudagraph_mode': <CUDAGraphMode.FULL_AND_PIECEWISE: (2, 1)>, 'cudagraph_num_of_warmups': 1, 'cudagraph_capture_sizes': [1,
 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 272, 288, 304, 320, 336, 352, 368, 384, 400, 416, 432, 448,
 464, 480, 496, 512], 'cudagraph_copy_inputs': False, 'cudagraph_specialize_lora': True, 'use_inductor_graph_partition': False, 'pass_config': {'fuse_norm_quant': False, 'fuse_act_quant': False, 'fuse_attn_quant':
 False, 'eliminate_noops': True, 'enable_sp': False, 'fuse_gemm_comms': False, 'fuse_allreduce_rms': False}, 'max_cudagraph_capture_size': 512, 'dynamic_shapes_config': {'type': <DynamicShapesType.BACKED: 'backed'
>, 'evaluate_guards': False}, 'local_cache_dir': None}
(EngineCore_DP0 pid=926472) INFO 03-15 16:15:13 [parallel_state.py:1203] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.16.44.23:40965 backend=nccl
(EngineCore_DP0 pid=926472) INFO 03-15 16:15:13 [parallel_state.py:1411] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=926472) INFO 03-15 16:15:14 [gpu_model_runner.py:3562] Starting to load model facebook/opt-125m...
(EngineCore_DP0 pid=926472) INFO 03-15 16:15:15 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
(EngineCore_DP0 pid=926472) INFO 03-15 16:15:15 [default_loader.py:308] Loading weights took 0.07 seconds
(EngineCore_DP0 pid=926472) INFO 03-15 16:15:16 [gpu_model_runner.py:3659] Model loading took 0.2389 GiB memory and 1.184900 seconds
(EngineCore_DP0 pid=926472) INFO 03-15 16:15:16 [decorators.py:432] Directly load AOT compilation from path /workspace/.cache/vllm/torch_aot_compile/d6c80550895aa1d4cac91b4fb9ef55dfdb1e6d77fa19cbd34edfd
5629fb0d0b1/rank_0_0/model
(EngineCore_DP0 pid=926472) INFO 03-15 16:15:18 [backends.py:643] Using cache directory: /workspace/.cache/vllm/torch_compile_cache/c9c272ba2f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=926472) INFO 03-15 16:15:18 [backends.py:703] Dynamo bytecode transform time: 2.46 s
(EngineCore_DP0 pid=926472) INFO 03-15 16:15:19 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 0.425 s
(EngineCore_DP0 pid=926472) INFO 03-15 16:15:19 [monitor.py:34] torch.compile takes 2.89 s in total
(EngineCore_DP0 pid=926472) INFO 03-15 16:15:19 [gpu_worker.py:375] Available KV cache memory: 126.00 GiB
(EngineCore_DP0 pid=926472) INFO 03-15 16:15:19 [kv_cache_utils.py:1291] GPU KV cache size: 3,669,872 tokens
(EngineCore_DP0 pid=926472) INFO 03-15 16:15:19 [kv_cache_utils.py:1296] Maximum concurrency for 2,048 tokens per request: 1791.93x
(EngineCore_DP0 pid=926472) INFO 03-15 16:15:21 [gpu_model_runner.py:4587] Graph capturing finished in 2 secs, took 0.04 GiB
(EngineCore_DP0 pid=926472) INFO 03-15 16:15:21 [core.py:259] init engine (profile, create kv cache, warmup model) took 5.97 seconds
INFO 03-15 16:15:22 [llm.py:360] Supported tasks: ['generate']

Model load time: 12.03 seconds
Generation time: 0.35 seconds

===== RESPONSE =====

 Joel Schulze explains.

Fautdig runesommer breven der G24bschesinhaber. Et swimförkeln no en diskussion sähte konferenzt:

Månad inntil tegsning föräldrar engagir. Domstamma folgt ett uppgäng enrNUMblokk för vatten – men vilket tal upp jobundersökt i moments dan:väldligen föresterna. Osäkerheten säger:

I kilometer gött billighert på alla lite, samtalen i nämnd arkeologistiskheten. Kan få betydningar? Give betydningministru. De har 8 år till Västerlo Institutet försökt att han går att egna biltamt:nejjande kara s
erioppalets person.

Och som Mo var från Yugoslavia såsom i Europa är det lite att säga att aldrig syvligt än vi somarant till Utveckling ”vat vilka betsägg är.” Vad som har medisinaare bild var?

Fö

===== RESPONSE END =====

ERROR 03-15 16:15:22 [core_client.py:606] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.

====================================================
RUNNING TEST WITH temperature=1.0 repeat=3
====================================================
===== TEST CONFIG =====
Model: facebook/opt-125m
Temperature: 1.0
Max tokens: 300
Repeat index: 3
Prompt: A researcher wants to use a large language model to help analyse scientific papers. What are three useful ways the model could help, and what are three risks the researcher should be careful about?
=======================
INFO 03-15 16:15:30 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'facebook/opt-125m'}
INFO 03-15 16:15:30 [model.py:514] Resolved architecture: OPTForCausalLM
INFO 03-15 16:15:30 [model.py:1661] Using max model len 2048
INFO 03-15 16:15:30 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=16384.
(EngineCore_DP0 pid=926844) INFO 03-15 16:15:31 [core.py:93] Initializing a V1 LLM engine (v0.13.0+faa43dbf.nv26.01) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/
opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parall
el_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutp
utsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_confi
g=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_metrics=False, kv_cache_metrics_sample=0.01, cudagraph_metrics=False, enable_layerwise_
nvtx_tracing=False), seed=0, served_model_name=facebook/opt-125m, enable_prefix_caching=True, enable_chunked_prefill=True, pooler_config=None, compilation_config={'level': None, 'mode': <CompilationMode.VLLM_COMPI
LE: 3>, 'debug_dump_path': None, 'cache_dir': '', 'compile_cache_save_format': 'binary', 'backend': 'inductor', 'custom_ops': ['none'], 'splitting_ops': ['vllm::unified_attention', 'vllm::unified_attention_with_ou
tput', 'vllm::unified_mla_attention', 'vllm::unified_mla_attention_with_output', 'vllm::mamba_mixer2', 'vllm::mamba_mixer', 'vllm::short_conv', 'vllm::linear_attention', 'vllm::plamo2_mamba_mixer', 'vllm::gdn_atte
ntion_core', 'vllm::kda_attention', 'vllm::sparse_attn_indexer'], 'compile_mm_encoder': False, 'compile_sizes': [], 'compile_ranges_split_points': [16384], 'inductor_compile_config': {'enable_auto_functionalized_v
2': False, 'combo_kernels': True, 'benchmark_combo_kernel': True}, 'inductor_passes': {}, 'cudagraph_mode': <CUDAGraphMode.FULL_AND_PIECEWISE: (2, 1)>, 'cudagraph_num_of_warmups': 1, 'cudagraph_capture_sizes': [1,
 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 272, 288, 304, 320, 336, 352, 368, 384, 400, 416, 432, 448,
 464, 480, 496, 512], 'cudagraph_copy_inputs': False, 'cudagraph_specialize_lora': True, 'use_inductor_graph_partition': False, 'pass_config': {'fuse_norm_quant': False, 'fuse_act_quant': False, 'fuse_attn_quant':
 False, 'eliminate_noops': True, 'enable_sp': False, 'fuse_gemm_comms': False, 'fuse_allreduce_rms': False}, 'max_cudagraph_capture_size': 512, 'dynamic_shapes_config': {'type': <DynamicShapesType.BACKED: 'backed'
>, 'evaluate_guards': False}, 'local_cache_dir': None}
(EngineCore_DP0 pid=926844) INFO 03-15 16:15:33 [parallel_state.py:1203] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.16.44.23:55277 backend=nccl
(EngineCore_DP0 pid=926844) INFO 03-15 16:15:33 [parallel_state.py:1411] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=926844) INFO 03-15 16:15:33 [gpu_model_runner.py:3562] Starting to load model facebook/opt-125m...
(EngineCore_DP0 pid=926844) INFO 03-15 16:15:34 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
(EngineCore_DP0 pid=926844) INFO 03-15 16:15:35 [default_loader.py:308] Loading weights took 0.08 seconds
(EngineCore_DP0 pid=926844) INFO 03-15 16:15:35 [gpu_model_runner.py:3659] Model loading took 0.2389 GiB memory and 1.200876 seconds
(EngineCore_DP0 pid=926844) INFO 03-15 16:15:35 [decorators.py:432] Directly load AOT compilation from path /workspace/.cache/vllm/torch_aot_compile/d6c80550895aa1d4cac91b4fb9ef55dfdb1e6d77fa19cbd34edfd
5629fb0d0b1/rank_0_0/model
(EngineCore_DP0 pid=926844) INFO 03-15 16:15:38 [backends.py:643] Using cache directory: /workspace/.cache/vllm/torch_compile_cache/c9c272ba2f/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=926844) INFO 03-15 16:15:38 [backends.py:703] Dynamo bytecode transform time: 2.43 s
(EngineCore_DP0 pid=926844) INFO 03-15 16:15:38 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 0.420 s
(EngineCore_DP0 pid=926844) INFO 03-15 16:15:38 [monitor.py:34] torch.compile takes 2.85 s in total
(EngineCore_DP0 pid=926844) INFO 03-15 16:15:39 [gpu_worker.py:375] Available KV cache memory: 126.00 GiB
(EngineCore_DP0 pid=926844) INFO 03-15 16:15:39 [kv_cache_utils.py:1291] GPU KV cache size: 3,669,872 tokens
(EngineCore_DP0 pid=926844) INFO 03-15 16:15:39 [kv_cache_utils.py:1296] Maximum concurrency for 2,048 tokens per request: 1791.93x
(EngineCore_DP0 pid=926844) INFO 03-15 16:15:41 [gpu_model_runner.py:4587] Graph capturing finished in 2 secs, took 0.04 GiB
(EngineCore_DP0 pid=926844) INFO 03-15 16:15:41 [core.py:259] init engine (profile, create kv cache, warmup model) took 6.01 seconds
INFO 03-15 16:15:42 [llm.py:360] Supported tasks: ['generate']

Model load time: 12.11 seconds
Generation time: 0.35 seconds

===== RESPONSE =====

 Joel Schulze explains.

Fautdig runesommer breven der G24bschesinhaber. Et swimförkeln no en diskussion sähte konferenzt:

Månad inntil tegsning föräldrar engagir. Domstamma folgt ett uppgäng enrNUMblokk för vatten – men vilket tal upp jobundersökt i moments dan:väldligen föresterna. Osäkerheten säger:

I kilometer gött billighert på alla lite, samtalen i nämnd arkeologistiskheten. Kan få betydningar? Give betydningministru. De har 8 år till Västerlo Institutet försökt att han går att egna biltamt:nejjande kara s
erioppalets person.

Och som Mo var från Yugoslavia såsom i Europa är det lite att säga att aldrig syvligt än vi somarant till Utveckling ”vat vilka betsägg är.” Vad som har medisinaare bild var?

Fö

===== RESPONSE END =====

ERROR 03-15 16:15:42 [core_client.py:606] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.

===== JOB END =====

## 4. Results

## Summary of behaviour

| Temperature | Behaviour | Consistency | Variation | Hallucination Risk |
|---|---|---|---|---|
| 0.0 | Repetitive and deterministic | Very high | None | Low, but output quality poor |
| 0.3 | Slightly more varied, but still repetitive | High | Low | Low to moderate |
| 0.7 | Noticeably unstable and less coherent | Medium | Moderate | Moderate |
| 1.0 | Highly unstable and largely nonsensical | Low | High | High |

## Detailed observations

### Temperature = 0.0

At temperature **0.0**, the model produced **nearly identical responses across runs**.

The repeated output was dominated by the same sentence pattern:

> “The researchers have been working on a new language model that could help analyse scientific papers.”

This was repeated many times with almost no meaningful development of the answer.

Observed behaviour:

- very high repeatability
- almost no variation across repeats
- poor usefulness despite consistency
- low hallucination in the sense of random invention, but still not a useful answer

This result shows that **deterministic output is not automatically good output**. The model was stable, but it was stably poor.

### Temperature = 0.3

At temperature **0.3**, the model began to vary slightly, but the outputs were still highly repetitive and low quality.

A repeated pattern appeared around phrases such as:

> “The model is based on a mathematical model of the universe.”

This was not relevant to the prompt and appears to be a form of drift or weak hallucination.

Observed behaviour:

- high repeatability across runs
- low but noticeable variation
- some irrelevant invented content
- poor usefulness for research tasks

This setting was only marginally more varied than `0.0`, but not meaningfully more helpful.

### Temperature = 0.7

At temperature **0.7**, the outputs became more unstable and significantly less coherent.

The model started producing confused text such as:

> “But the computer is not a computer. What is the computer model?”

This indicates stronger drift and breakdown of explanation quality.

Observed behaviour:

- more variation than at lower temperatures
- much less coherence
- more obvious hallucination-like behaviour
- weak usefulness for research support

This temperature introduced more diversity, but the small model could not use that freedom productively.

### Temperature = 1.0

At temperature **1.0**, the output degraded further and became largely nonsensical.

One output began with:

> “Joel Schulze explains.”

and then continued into text that mixed fragments of different languages and meaningless phrases.

Observed behaviour:

- very high variation
- very low coherence
- strong hallucination / nonsense generation
- not useful for research tasks

This confirms that at high temperature, a small model can quickly become unusable for serious explanatory work.

## 5. Interpretation

This experiment shows the expected relationship between **temperature and output stability**, but also highlights something more important: **model quality places a hard limit on how useful temperature tuning can be**.

### Main pattern observed

- At **low temperature**, the model was highly repeatable, but the answer was repetitive and poor.
- At **medium temperature**, the model showed slightly more variation, but still produced confused and weak responses.
- At **high temperature**, the outputs became increasingly unstable, hallucinated, and eventually nonsensical.

### What this means

Temperature changed the **style and variability** of the output, but it did not solve the underlying limitation of the model.

For this particular setup:

- `0.0` gave the most consistent output
- `0.3` added slight variation
- `0.7` increased drift
- `1.0` produced clear breakdown

### Practical implications for research workflows

For HPC-based LLM workflows, temperature should be chosen based on the task:

| Use case | Recommended temperature |
|---|---|
| Reproducible experiments | 0.0 |
| Technical explanations | 0.2 – 0.4 |
| General reasoning tasks | 0.5 – 0.7 |
| Creative exploration | 0.8 – 1.0 |

However, this test also shows that **temperature tuning is only meaningful when the base model is capable enough for the task**. A very small model like `facebook/opt-125m` is useful for pipeline testing, but not for reliable research support.

### Key takeaway

Temperature strongly affects:

- repeatability
- variation
- hallucination risk
- usefulness of the output

But temperature alone cannot compensate for weak model capability.

For future work, this same test should be repeated with a stronger instruct model so that the effect of temperature can be observed without the results being dominated by model weakness.